# M10. 파생변수 만들기

> 📌 **언제 필요한가**  
> 기존 컬럼들을 조합해서 새로운 의미 있는 변수를 만들 때.  
> 예: 진학률 = 진학자 ÷ 졸업자 × 100

## 이 모듈에서 배울 것

- 사칙연산으로 파생변수
- 조건문(`np.where`, `apply`)으로 카테고리 파생변수
- 비율, 차이, 누적 같은 패턴
- 작년 선배(김서윤) 사례: 진학률 계산

---


## 📥 데이터 준비

> 이 모듈은 아래 파일이 필요해요. **저장소에는 동봉돼 있지 않으니** 먼저 받아서 두세요.
> 받는 곳 링크를 누르면 바로 받으러 갈 수 있어요.

- `한국교육개발원_시도 시군구별 졸업자 진학자 진학률_20240401.csv` — 인코딩 `cp949` — [공공데이터포털에서 받기](https://www.data.go.kr/data/15053808/fileData.do)
- `교육부_시도별 진로전담교사 배치현황_20241231.csv` — 인코딩 `cp949` — [공공데이터포털에서 받기](https://www.data.go.kr/data/15097012/fileData.do)

두는 곳 — **로컬 Jupyter**: 이 노트북과 같은 폴더 / **Colab**: `/content/`에 업로드.  
컬럼 설명·함정 등 자세한 내용은 [`data/README.md`](data/README.md) 참고.

---

## 1. 가장 단순한 파생변수 — 사칙연산

작년 선배(김서윤)가 한 진학률 계산 그대로 재현.


In [ ]:
import pandas as pd

graduate = pd.read_csv('data/한국교육개발원 시도 시군구별 졸업자 진학자 진학률_20240401.csv', encoding='cp949')

# 시도별로 집계
graduate_by_sido = graduate.groupby('시도')[['졸업자', '진학자']].sum().reset_index()
graduate_by_sido.head()


In [ ]:
# 진학률 = 진학자 / 졸업자 × 100  ← 파생변수!
graduate_by_sido['진학률'] = graduate_by_sido['진학자'] / graduate_by_sido['졸업자'] * 100

# 진학률 높은 순
graduate_by_sido.sort_values('진학률', ascending=False).head()


> 💡 한 줄로 끝나요. 두 컬럼 연산 → 새 컬럼.  
> 이게 가장 흔한 파생변수 패턴.


## 2. 자주 만드는 파생변수 패턴

### 2.1 비율 / 퍼센티지


In [ ]:
# 위에서 한 진학률이 비율 파생변수
# 일반 패턴: A / B 또는 A / B * 100


### 2.2 차이


In [ ]:
# 합격자 = 졸업자 - 미합격자 (예시)
graduate_by_sido['미진학자'] = graduate_by_sido['졸업자'] - graduate_by_sido['진학자']
graduate_by_sido.head()


### 2.3 누적합 (시계열)


In [ ]:
# 연도별 합계로 누적 (예시)
yearly = graduate.groupby('연도')['졸업자'].sum().reset_index()
yearly['누적졸업자'] = yearly['졸업자'].cumsum()
yearly


### 2.4 변화량 / 변화율


In [ ]:
yearly['전년대비_변화'] = yearly['졸업자'].diff()
yearly['전년대비_변화율(%)'] = yearly['졸업자'].pct_change() * 100
yearly


## 3. 조건부 파생변수 — `np.where`


In [ ]:
import numpy as np

# 진학률 75% 이상이면 '높음', 아니면 '낮음'
graduate_by_sido['진학률_등급'] = np.where(
    graduate_by_sido['진학률'] >= 75,
    '높음',
    '낮음'
)
graduate_by_sido[['시도', '진학률', '진학률_등급']].head()


## 4. 여러 조건 — `pd.cut` 또는 `apply`


In [ ]:
# 구간으로 나누기 (pd.cut)
graduate_by_sido['진학률_구간'] = pd.cut(
    graduate_by_sido['진학률'],
    bins=[0, 70, 75, 80, 100],
    labels=['하', '중하', '중상', '상']
)
graduate_by_sido[['시도', '진학률', '진학률_구간']].head()


In [ ]:
# 복잡한 로직은 apply
def classify(rate):
    if rate >= 80:
        return '최상위'
    elif rate >= 75:
        return '상위'
    elif rate >= 70:
        return '평균'
    else:
        return '하위'

graduate_by_sido['진학률_분류'] = graduate_by_sido['진학률'].apply(classify)
graduate_by_sido[['시도', '진학률', '진학률_분류']].head()


## 5. 다른 데이터와 결합 후 파생변수


In [ ]:
# 진로전담교사 데이터와 합쳐서 "교사 1명당 졸업자" 계산
counselor = pd.read_csv('data/교육부_시도별 진로전담교사 배치현황_20241231.csv', encoding='cp949')

combined = pd.merge(graduate_by_sido, counselor[['시도', '진로전담교사수']], on='시도')
combined['교사1인당_졸업자'] = combined['졸업자'] / combined['진로전담교사수']
combined.sort_values('교사1인당_졸업자', ascending=False).head()


## 6. 본인 데이터에 적용해보기 ✏️


In [ ]:
# 본인 데이터로 의미 있는 파생변수 만들기
# 
# # 예 1: 비율
# my_df['비율'] = my_df['일부'] / my_df['전체'] * 100
# 
# # 예 2: 차이
# my_df['차이'] = my_df['올해'] - my_df['작년']
# 
# # 예 3: 조건 분류
# import numpy as np
# my_df['등급'] = np.where(my_df['점수'] >= 80, '우수', '보통')


## 7. ⚠️ 함정 / 주의사항

### 7.1 0으로 나누기
`A / B`에서 B가 0이면 결과가 `inf` 또는 NaN.  
**해결**:
```python
df['비율'] = df['A'] / df['B'].replace(0, np.nan)
```

### 7.2 변수명 충돌
이미 있는 컬럼명에 새 값을 할당하면 덮어써짐.  
**해결**: 새 이름 사용 또는 명시적 의도.

### 7.3 파생변수 의미 기록
다른 사람이 코드를 봤을 때 이해 가능하게 변수명을 명확히.  
`ratio` 대신 `진학률` 같은 명시적 이름.


## 8. 📚 더 알아보기

- `df.assign(new_col=...)` — 메서드 체이닝 친화적
- `df.eval('new_col = A + B')` — 문자열로 표현식
- `df.rolling(7).mean()` — 이동평균
- `df.shift(1)` — 행 시프트 (전일 값 등)
